# 02 — Pro Pattern Analysis
Cluster Challenger comps into archetypes. Analyze augment and item win rates per comp.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import pickle

sns.set_theme(style='darkgrid')
df = pd.read_csv('../data/processed/match_features.csv')
aug_df = pd.read_csv('../data/processed/augment_winrates.csv')
item_df = pd.read_csv('../data/processed/item_winrates.csv')
print(f'Matches: {df.shape[0]:,} participants | Augment rows: {aug_df.shape[0]:,} | Item rows: {item_df.shape[0]:,}')

In [ ]:
# Load K-Means clusters and assign to df
with open('../models/kmeans.pkl', 'rb') as f:
    km = pickle.load(f)

trait_enc = km['le_trait'].transform(
    df['top_trait'].fillna('unknown').apply(lambda x: x if x in km['le_trait'].classes_ else 'unknown')
)
numeric_vec = km['scaler'].transform(df[['level_efficiency', 'unit_star_avg', 'active_trait_count']].fillna(0))
cluster_input = np.hstack([numeric_vec, trait_enc.reshape(-1,1)])
df['cluster'] = km['kmeans'].predict(cluster_input)

cluster_profiles = (
    df.groupby('cluster')
    .agg(
        games=('placement','count'),
        top4_rate=('top4','mean'),
        avg_placement=('placement','mean'),
        top_trait=('top_trait', lambda x: x.value_counts().index[0]),
        avg_level_eff=('level_efficiency','mean'),
        avg_gold_left=('gold_left','mean'),
    )
    .sort_values('top4_rate', ascending=False)
)
cluster_profiles

In [ ]:
# PCA visualization of clusters
pca = PCA(n_components=2)
coords = pca.fit_transform(cluster_input)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(coords[:, 0], coords[:, 1], c=df['cluster'], cmap='tab20', alpha=0.4, s=10)
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.set_title('Comp Clusters (PCA projection)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

In [ ]:
# Top augments per comp
top_comps = df['top_trait'].value_counts().head(5).index.tolist()
for comp in top_comps:
    sub = aug_df[aug_df['top_trait'] == comp].head(5)
    if sub.empty: continue
    print(f'\n=== {comp} — Top Augments ===')
    print(sub[['augment','games','top4_rate','avg_placement']].to_string(index=False))

In [ ]:
# Top item combos for the 10 most played carries
top_units = item_df.groupby('unit_id')['games'].sum().nlargest(10).index
for unit in top_units:
    sub = item_df[item_df['unit_id'] == unit].head(3)
    print(f'\n{unit}:')
    print(sub[['item_combo','games','top4_rate','avg_placement']].to_string(index=False))